# Bitcoin Price Prediction - Quickstart Guide

This notebook demonstrates how to use the BTC Prediction system to:
1. Collect historical Bitcoin data
2. Create technical indicators and features
3. Train prediction models (LSTM, GRU, XGBoost)
4. Make predictions and visualize results

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collector import BTCDataCollector
from src.feature_engineering import create_features
from src.preprocessing import prepare_data_for_lstm, prepare_data_for_xgboost
from models.lstm_model import create_lstm_model
from models.gru_model import create_gru_model
from models.xgboost_model import create_xgboost_model

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 6)

print("Imports successful!")

## 1. Data Collection

First, let's collect historical Bitcoin price data.

In [ ]:
# Initialize data collector
collector = BTCDataCollector()

# Fetch 90 days of hourly data
df_raw = collector.fetch_yahoo_finance(days=90, interval='1h')

print(f"Data shape: {df_raw.shape}")
print(f"\nDate range: {df_raw['date'].min()} to {df_raw['date'].max()}")
print(f"\nFirst few rows:")
df_raw.head()

In [ ]:
# Plot raw price data
plt.figure(figsize=(15, 6))
plt.plot(df_raw['date'], df_raw['close'], linewidth=1.5)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.title('Bitcoin Price (Close)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Feature Engineering

Create technical indicators and features.

In [ ]:
# Create all features
df = create_features(df_raw, feature_set='all')

print(f"Data shape with features: {df.shape}")
print(f"\nNumber of features: {len(df.columns)}")
print(f"\nFeature columns:")
print(df.columns.tolist())

In [ ]:
# Visualize some technical indicators
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Price with moving averages
axes[0].plot(df['date'], df['close'], label='Close Price', linewidth=1.5)
axes[0].plot(df['date'], df['sma_7'], label='SMA 7', linewidth=1, alpha=0.7)
axes[0].plot(df['date'], df['sma_30'], label='SMA 30', linewidth=1, alpha=0.7)
axes[0].set_ylabel('Price (USD)')
axes[0].set_title('Price with Moving Averages')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df['date'], df['rsi_14'], label='RSI', color='purple', linewidth=1.5)
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold')
axes[1].set_ylabel('RSI')
axes[1].set_title('Relative Strength Index (RSI)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Volume
axes[2].bar(df['date'], df['volume'], alpha=0.5, color='steelblue')
axes[2].set_ylabel('Volume')
axes[2].set_xlabel('Date')
axes[2].set_title('Trading Volume')
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Train LSTM Model

Train an LSTM model for price prediction.

In [ ]:
# Prepare data for LSTM
data_lstm = prepare_data_for_lstm(
    df,
    sequence_length=60,
    forecast_horizon=1,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
)

print(f"Training data shape: {data_lstm['X_train'].shape}")
print(f"Validation data shape: {data_lstm['X_val'].shape}")
print(f"Test data shape: {data_lstm['X_test'].shape}")

In [ ]:
# Create and train LSTM model
lstm_model = create_lstm_model(
    sequence_length=data_lstm['sequence_length'],
    n_features=data_lstm['n_features'],
    model_type='lstm',
    lstm_units=[128, 64, 32],
    dropout_rate=0.2,
    learning_rate=0.001
)

# Build model
lstm_model.build_model()
lstm_model.get_model_summary()

# Train (reduce epochs for quick demo)
history = lstm_model.train(
    data_lstm['X_train'],
    data_lstm['y_train'],
    data_lstm['X_val'],
    data_lstm['y_val'],
    epochs=50,  # Increase for better results
    batch_size=32,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('LSTM Training Loss')
axes[0].legend()
axes[0].grid(True)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('LSTM Training MAE')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
metrics = lstm_model.evaluate(data_lstm['X_test'], data_lstm['y_test'])

print("Test Set Performance:")
print(f"  MAE:  {metrics['mae']:.6f}")
print(f"  RMSE: {metrics['rmse']:.6f}")
print(f"  MAPE: {metrics['mape']:.2f}%")
print(f"  Directional Accuracy: {metrics['directional_accuracy']:.2%}")

## 4. Train XGBoost Model

Train an XGBoost model for comparison.

In [ ]:
# Prepare data for XGBoost
data_xgb = prepare_data_for_xgboost(
    df,
    forecast_horizon=1,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15
)

print(f"Training data shape: {data_xgb['X_train'].shape}")
print(f"Number of features: {data_xgb['n_features']}")

In [ ]:
# Create and train XGBoost model
xgb_model = create_xgboost_model(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.01
)

xgb_model.train(
    data_xgb['X_train'],
    data_xgb['y_train'],
    data_xgb['X_val'],
    data_xgb['y_val'],
    early_stopping_rounds=50,
    verbose=True
)

In [ ]:
# Evaluate XGBoost model
metrics_xgb = xgb_model.evaluate(data_xgb['X_test'], data_xgb['y_test'])

print("Test Set Performance:")
print(f"  MAE:  {metrics_xgb['mae']:.6f}")
print(f"  RMSE: {metrics_xgb['rmse']:.6f}")
print(f"  MAPE: {metrics_xgb['mape']:.2f}%")
print(f"  Directional Accuracy: {metrics_xgb['directional_accuracy']:.2%}")

In [ ]:
# Plot feature importance
feature_importance = xgb_model.get_feature_importance(data_xgb['feature_columns'])

# Get top 20 features
top_features = dict(list(feature_importance.items())[:20])

plt.figure(figsize=(12, 8))
plt.barh(list(top_features.keys()), list(top_features.values()), color='steelblue')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title('Top 20 Most Important Features (XGBoost)', fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 5. Model Comparison

Compare the performance of different models.

In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Model': ['LSTM', 'XGBoost'],
    'MAE': [metrics['mae'], metrics_xgb['mae']],
    'RMSE': [metrics['rmse'], metrics_xgb['rmse']],
    'MAPE': [metrics['mape'], metrics_xgb['mape']],
    'Directional Accuracy': [metrics['directional_accuracy'], metrics_xgb['directional_accuracy']]
})

print("Model Comparison:")
print(comparison.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics_to_plot = ['MAE', 'RMSE', 'MAPE', 'Directional Accuracy']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    values = comparison[metric].values
    models = comparison['Model'].values
    
    ax.bar(models, values, color=['#1f77b4', '#ff7f0e'])
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} Comparison')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Save Models

Save the trained models for later use.

In [ ]:
# Save LSTM model
lstm_model.save_model('../models/trained/lstm_model_quickstart.h5')
data_lstm['preprocessor'].save_scalers('../models/trained/lstm_preprocessor_quickstart.pkl')

# Save XGBoost model
xgb_model.save_model('../models/trained/xgboost_model_quickstart.pkl')

print("Models saved successfully!")

## Next Steps

1. Try different model architectures and hyperparameters
2. Experiment with different feature sets
3. Add more advanced features like sentiment analysis
4. Implement ensemble methods combining multiple models
5. Create a real-time prediction system

For more details, see the documentation and other example notebooks.